# 00 · Pack checkpoints for Kaggle (no GPU needed)
Run this on Colab with **"Connect without GPU"**. It puts the trained EMA weights (A, B, C, D, if
present) and their loss logs into one small zip on your Drive: `MyDrive/fastdiff-med/kaggle_upload.zip`
(about 100 MB; weights are stored in half precision). Download that zip and upload it to Kaggle as a
**new dataset** (any name). The Kaggle notebook finds it automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, zipfile, io, torch
BASE = '/content/drive/MyDrive/fastdiff-med'
out = f'{BASE}/kaggle_upload.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
    for run in 'ABCD':
        p = f'{BASE}/checkpoints/{run}/ema.pt'
        if not os.path.exists(p):
            print(f'{run}: no ema.pt (skipped)'); continue
        ck = torch.load(p, map_location='cpu', weights_only=False)
        ck['model'] = {k: (v.half() if v.is_floating_point() else v) for k, v in ck['model'].items()}
        buf = io.BytesIO(); torch.save(ck, buf)
        z.writestr(f'checkpoints/{run}/ema.pt', buf.getvalue())
        if os.path.exists(f'{BASE}/checkpoints/{run}/loss.csv'):
            z.write(f'{BASE}/checkpoints/{run}/loss.csv', f'checkpoints/{run}/loss.csv')
        print(f"{run}: packed (trained {ck['config'].get('train_steps', '?')} steps)")
print(f'wrote {out}: {os.path.getsize(out) / 1e6:.0f} MB')